In [ ]:
# pip install langchain langchain-openai requests

In [1]:
import langchain
import requests
import importlib.metadata

print(f"LangChain version: {langchain.__version__}")
print(f"Requests version: {requests.__version__}")

try:
    l_openai_version = importlib.metadata.version("langchain-openai")
    print(f"LangChain-OpenAI version: {l_openai_version}")
except importlib.metadata.PackageNotFoundError:
    print("LangChain-OpenAI is not installed.")


LangChain version: 1.2.15
Requests version: 2.33.1
LangChain-OpenAI version: 1.2.1


In [2]:
# Ideally put this in your enviroment

OPENAI_API_KEY = ""

In [5]:
import math
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

# TOOL 1: EMI Calculator
@tool
def calculate_emi(principal: float, rate: float, time: float) -> str:
    """
    Calculate EMI for a loan.

    Args:
        principal: Loan amount
        rate: Annual interest rate (in %)
        time: Loan duration in years
    """
    print("...Running EMI calculator tool")

    try:
        R = rate / 12 / 100   # monthly rate
        N = time * 12         # months
        emi = (principal * R * (1 + R)**N) / ((1 + R)**N - 1)
        return f"Monthly EMI is {round(emi, 2)}"

    except:
        return "Error calculating EMI"


# TOOL 2: Simple Interest
@tool
def calculate_simple_interest(principal: float, rate: float, time: float) -> str:
    """
    Calculate simple interest.

    Args:
        principal: Initial amount
        rate: Annual interest rate (in %)
        time: Time in years
    """
    print("...Running simple interest tool")

    try:
        SI = (principal * rate * time) / 100
        return f"Simple Interest is {round(SI, 2)}"

    except:
        return "Error calculating simple interest"


In [6]:
tools = [calculate_emi, calculate_simple_interest]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant with access to tools."
)

## Try
```
input1: "I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?"

input2: "I want EMI for a loan of 200000 at 10 percent for 2 years ?"

input3_SPELL_ERROR: "What is the simpl interst on 5000 at 5 percent for 3 years ?" 

input4: "I am living in Moscow. The simple iterest is 5% there on loans. 
         I took a loan on simple interest for 3 yeas . The loan amt. was 5000. 
         What is the toal simple interest I would end up paying ?"

In [7]:
user_input = "I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running EMI calculator tool
Response: {'messages': [HumanMessage(content="I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?", additional_kwargs={}, response_metadata={}, id='152fb727-1750-42f6-9a53-e13e9f179834'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 157, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9987326bf5', 'id': 'chatcmpl-DZsHVYDBfHERhErdib03JLSwp6Ozq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7d4-4c53-7d20-88ff-f67215072183-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 200000, 'rate': 10, 'time': 2}, 'id': 'call_3CpDzQ1VmNcVNsxo

In [8]:
user_input = "I want EMI for a loan of 200000 at 10 percent for 2 years ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running EMI calculator tool
Response: {'messages': [HumanMessage(content='I want EMI for a loan of 200000 at 10 percent for 2 years ?', additional_kwargs={}, response_metadata={}, id='1e556a1c-2373-4259-9a0f-180b477c4453'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 155, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f266aff3a9', 'id': 'chatcmpl-DZsHou3qS7nWQYAhq4qGW9UpQj2h1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7d4-9314-7882-9eaa-beed060d184c-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 200000, 'rate': 10, 'time': 2}, 'id': 'call_Z5X5KDXONAjWIWD5

In [9]:
user_input = "What is the simpl interst on 5000 at 5 percent for 3 years ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running simple interest tool
Response: {'messages': [HumanMessage(content='What is the simpl interst on 5000 at 5 percent for 3 years ?', additional_kwargs={}, response_metadata={}, id='c8357ee7-3ed4-49fa-b79f-54519a162806'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 155, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9987326bf5', 'id': 'chatcmpl-DZsJWYFYfHMfrBsexQBOkaBJ1ceov', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7d6-3290-7db0-bc85-316cee733d5b-0', tool_calls=[{'name': 'calculate_simple_interest', 'args': {'principal': 5000, 'rate': 5, 'time': 3}, 'id': 'call_gk3iq

In [12]:
user_input = "I am living in Moscow. The simple iterest is 5% there on loans.\
              I took a loan on simple interest for 3 yeas . The loan amt. was 5000 . \
              What is the toal simple interest I would end up paying ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running simple interest tool
Response: {'messages': [HumanMessage(content='I am living in Moscow. The simple iterest is 5% there on loans.              I took a loan on simple interest for 3 yeas . The loan amt. was 5000 .               What is the toal simple interest I would end up paying ?', additional_kwargs={}, response_metadata={}, id='3d073e0f-ef15-4dca-94a4-f47b16793061'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 190, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9987326bf5', 'id': 'chatcmpl-DZsLIBB8QrKjYYzyA7W2yuUVgfJPM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0